# 1 - Téléchargement et préparation Intel Image Classification

Ce notebook télécharge **Intel Image Classification** depuis Kaggle
(`puneet6060/intel-image-classification`) et prépare la structure multi-fidélité :

- `test/` : jeu de test (seg_test fourni par Kaggle, ~3 000 images)
- `train/HF/` : 10 % des images d'entraînement (Haute Fidélité)
- `train/BF/` : 90 % des images d'entraînement (Basse Fidélité, dégradées à la volée)

**Classes (6)** : buildings, forest, glacier, mountain, sea, street — ~25 000 images au total

**Prérequis Colab** : dans l'onglet `Secrets`, ajouter `KAGGLE_USERNAME` et `KAGGLE_KEY`
(disponibles sur kaggle.com → votre compte → API → Create New Token).

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

if env_config.in_colab():
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Credentials Kaggle configures depuis les secrets Colab.')
    except Exception as e:
        print(f'Secrets non disponibles : {e}')
        print('Alternative : uploade ~/.kaggle/kaggle.json manuellement dans Colab.')
    get_ipython().system('pip install -q kagglehub')
    import kagglehub
    INTEL_RAW_PATH = kagglehub.dataset_download('puneet6060/intel-image-classification')
else:
    INTEL_RAW_PATH = os.path.join(env_config.project_home(), 'data', 'Intel', 'raw_download')
    os.makedirs(INTEL_RAW_PATH, exist_ok=True)
    print(f'Local : place les fichiers seg_train/ et seg_test/ dans {INTEL_RAW_PATH}')
print('INTEL_RAW_PATH:', INTEL_RAW_PATH)

In [ ]:
import sys, os, importlib, shutil, random
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

def find_subdir(base, name):
    """Trouve le dossier de classes (gere la structure doublee seg_train/seg_train/)."""
    for candidate in [os.path.join(base, name, name), os.path.join(base, name)]:
        if os.path.isdir(candidate) and any(
                os.path.isdir(os.path.join(candidate, d)) for d in os.listdir(candidate)):
            return candidate
    raise FileNotFoundError(f'Impossible de trouver "{name}" dans {base}')

train_src = find_subdir(INTEL_RAW_PATH, 'seg_train')
test_src  = find_subdir(INTEL_RAW_PATH, 'seg_test')
print('Train source:', train_src)
print('Test source :', test_src)
classes = sorted(os.listdir(train_src))
print('Classes     :', classes)

out_dir = env_config.data_dir('Intel')
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
for d in [f'{out_dir}/test', f'{out_dir}/train/HF', f'{out_dir}/train/BF']:
    os.makedirs(d, exist_ok=True)

for cls in classes:
    src_test_cls = os.path.join(test_src, cls)
    if os.path.isdir(src_test_cls):
        shutil.copytree(src_test_cls, os.path.join(out_dir, 'test', cls))
    for split in ['HF', 'BF']:
        os.makedirs(os.path.join(out_dir, 'train', split, cls), exist_ok=True)
    images = [f for f in os.listdir(os.path.join(train_src, cls))
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42)
    random.shuffle(images)
    split_idx = int(len(images) * 0.10)
    for n in images[:split_idx]:
        shutil.copy2(os.path.join(train_src, cls, n),
                     os.path.join(out_dir, 'train', 'HF', cls, n))
    for n in images[split_idx:]:
        shutil.copy2(os.path.join(train_src, cls, n),
                     os.path.join(out_dir, 'train', 'BF', cls, n))

for split_name in ['test', 'train/HF', 'train/BF']:
    total = sum(len(os.listdir(os.path.join(out_dir, split_name, c)))
                for c in classes if os.path.isdir(os.path.join(out_dir, split_name, c)))
    print(f'{split_name:12s}: {total} images')
print('Termine ! Intel multi-fidelite dans:', out_dir)

In [ ]:
import sys, os, importlib, shutil
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Sur Colab : zipper + uploader sur le Drive. Sur serveur/local : rien a faire.
if env_config.in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = '/content/drive/MyDrive/UTBM_PF22/datasets/Intel'
    os.makedirs(drive_path, exist_ok=True)
    get_ipython().system('cd /content && zip -0 -r -q intel_multifidelity.zip processed_multifidelity/')
    shutil.copy2('/content/intel_multifidelity.zip', f'{drive_path}/dataset_multifidelity.zip')
    print('Zip Intel uploade sur le Drive.')
else:
    print('Serveur/local : pas de zip necessaire (donnees deja sur le disque).')